# Silicon current-share and pathway-dependent eSOH exports

This notebook exports the intermediate states used to construct the silicon current-share analysis. 

It maps material-level silicon and graphite OCP derivatives onto the full-cell SoC axis using pathway-specific eSOH parameters, then saves the material-resolved dQ/dV, silicon current share, effective silicon OCP, and lithiation-capacity trajectories used in the figure panels.

In [ ]:
from __future__ import annotations

import os
import sys
import tempfile
from pathlib import Path


def find_repo_root(start=None):
    """Find the repository root from the current working directory."""
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "code" / "diagnostic_algorithm_lifetime_crate").is_dir() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not find the repository root. Run this notebook from inside the cloned repository.")


REPO_ROOT = find_repo_root()
CODE_DIR = REPO_ROOT / "code"
NOTEBOOK_DIR = CODE_DIR / "pathway_dependent_evolution"
OUTPUT_DIR = NOTEBOOK_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CACHE_DIR = Path(tempfile.gettempdir()) / "pathway_dependent_evolution_cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(CACHE_DIR / "matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", str(CACHE_DIR / "xdg"))

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter

print("REPO_ROOT  =", REPO_ROOT)
print("OUTPUT_DIR =", OUTPUT_DIR)


In [ ]:
from diagnostic_algorithm_lifetime_crate.user_functions import (
    NMC_OCP,
    Gr_OCP,
    Si_OCP,
    build_effective_si_ocp,
    ocp_3,
)
from diagnostic_algorithm_lifetime_crate.derivative_utils import smooth_then_grad


# Pathway-dependent eSOH cases

Each row below defines one eSOH scenario used to compare BOL behavior, isolated degradation pathways, and measured/mixed EOL degradation. The internal parameter order is `[Cn_Si, Cn_Gr, x100, Cp, y100, si_scale_a, si_shift_b]`, corresponding to the manuscript notation `[C_n,Si, C_n,Gr, x_n,100, C_p, x_p,100, s_V, U_off]`. The first five values set the material-specific eSOH, and the last two define the effective silicon-OCP deformation.

In [ ]:
# Common capacity grid and Savitzky-Golay settings for V(Q) and derivative traces.
Qdata_expand = np.linspace(-3, 3, 900)
win = 13
poly = 3

# eSOH case parameters use the saved-output schema:
# [Cn_Si, Cn_Gr, x100, Cp, y100, si_scale_a, si_shift_b]
# = [C_n,Si, C_n,Gr, x_n,100, C_p, x_p,100, s_V, U_off] in the manuscript.
# Cases are ordered to match the pathway-dependent comparison in the paper.
optimized_res_list = [
    [1.1200, 1.4700, 0.9700, 2.7000, 0.0210, 0.9100, 0.0290],
    [1.2300, 1.1800, 0.71, 2.7400, 0.0210, 0.9100, 0.0400],
    [0.3400, 1.4700, 0.9700, 2.7000, 0.0210, 0.44, 0.08,],
    [1.1200, 0.5300, 0.9700, 2.7000, 0.0210, 0.9100, 0.0290],
    [1.1200, 1.4700, 0.9700, 1.8800, 0.0210, 0.9100, 0.0290],
    [0.456082777, 1.31181201, 0.986295044, 2.725352537, 0.023007387, 0.515796598, 0.08],
    [0.3400, 1.4700, 0.9700, 2.7000, 0.0210, 0.9100, 0.0290,],
]
labels = ["BOL", "LLI", "LAM Si", "LAM NMC", "LAM Gr", "CELL20EOL", "LAM Si no deform"]

SI_RECONSTRUCTION_MODE = "reconstructed"

# Notebook-generated CSV outputs are separated from MATLAB-rendered figures.
eSOH_export_DIR = OUTPUT_DIR / "data"
eSOH_export_DIR.mkdir(parents=True, exist_ok=True)



# Export material-resolved current-share states

For each eSOH scenario, this section reconstructs the cell voltage and electrode potentials, maps silicon and graphite lithiation onto the common full-cell SoC axis, computes material-resolved dQ/dV contributions, and saves the CSV files consumed by the MATLAB plotting script.

In [ ]:
# Potential windows used to keep physically meaningful electrode traces.
CATHODE_P_MIN = 3.53
CATHODE_P_MAX = 4.2900
ANODE_P_MIN   = 0.0375
ANODE_P_MAX   = 0.999

# Helper functions for interpolation, derivative inversion, and masking.


def _unique_sorted_xy(x, y):
    x = np.asarray(x, float).reshape(-1)
    y = np.asarray(y, float).reshape(-1)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]
    if x.size == 0:
        return x, y
    order = np.argsort(x)
    x = x[order]
    y = y[order]
    x_u, idx = np.unique(x, return_index=True)
    y_u = y[idx]
    return x_u, y_u

def inv_or_nan(arr, eps=1e-12):
    arr = np.asarray(arr, float)
    out = np.full_like(arr, np.nan, dtype=float)
    m = np.isfinite(arr) & (np.abs(arr) > eps)
    out[m] = 1.0 / arr[m]
    return out

def apply_nan_mask(arr, mask):
    arr = np.asarray(arr, float).copy()
    arr[~mask] = np.nan
    return arr

def build_si_inverse_map(si_scale_a, si_shift_b, reconstruction_mode="direct"):
    si_ocp_eff = build_effective_si_ocp(
        si_scale_a=si_scale_a,
        si_shift_b=si_shift_b,
        reconstruction_mode=reconstruction_mode,
    )
    x_si = np.asarray(si_ocp_eff["sto"], float)
    p_si = np.asarray(si_ocp_eff["p"], float)

    order = np.argsort(p_si)
    p_si = p_si[order]
    x_si = x_si[order]
    p_si, idx = np.unique(p_si, return_index=True)
    x_si = x_si[idx]

    f_si_sto = interp1d(
        p_si,
        x_si,
        bounds_error=False,
        fill_value=np.nan,
    )
    return si_ocp_eff, f_si_sto

# Smooth graphite OCP before inverse mapping U_anode -> x_gr.


def smooth_ocp_curve(
    df,
    x_col="sto",
    y_col="p",
    window_length=31,
    polyorder=3,
    enforce_monotone=True,
):
    x = np.asarray(df[x_col], float)
    y = np.asarray(df[y_col], float)

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    # Sort by stoichiometry and remove duplicate x values.
    order = np.argsort(x)
    x = x[order]
    y = y[order]

    x_u, idx = np.unique(x, return_index=True)
    y_u = y[idx]

    # Enforce monotone-decreasing lithiation OCP before smoothing.
    if enforce_monotone:
        y_mono = np.minimum.accumulate(y_u)
    else:
        y_mono = y_u.copy()

    # Smooth the monotone OCP while preserving the valid support.
    w = min(window_length, len(x_u) if len(x_u) % 2 == 1 else len(x_u) - 1)
    if w < polyorder + 3:
        w = polyorder + 3
    if w % 2 == 0:
        w += 1
    if w > len(x_u):
        w = len(x_u) if len(x_u) % 2 == 1 else len(x_u) - 1

    y_smooth = savgol_filter(y_mono, window_length=w, polyorder=polyorder, mode="interp")

    # Re-enforce monotonicity after smoothing.
    if enforce_monotone:
        y_smooth = np.minimum.accumulate(y_smooth)

    dUdx_smooth = np.gradient(y_smooth, x_u)

    return pd.DataFrame({
        "sto": x_u,
        "p": y_u,                 # raw
        "p_monotone": y_mono,     # after monotone projection
        "p_smooth": y_smooth,     # after smoothing
        "dUdx_smooth": dUdx_smooth,
    })

Gr_OCP_smooth = smooth_ocp_curve(
    Gr_OCP,
    x_col="sto",
    y_col="p",
    window_length=51,
    polyorder=3,
    enforce_monotone=True,
)

def build_gr_inverse_map():
    x_gr = np.asarray(Gr_OCP_smooth["sto"], float)
    p_gr = np.asarray(Gr_OCP_smooth["p_smooth"], float)

    order = np.argsort(p_gr)
    p_gr = p_gr[order]
    x_gr = x_gr[order]
    p_gr, idx = np.unique(p_gr, return_index=True)
    x_gr = x_gr[idx]

    f_gr_sto = interp1d(
        p_gr,
        x_gr,
        bounds_error=False,
        fill_value=np.nan,
    )
    return f_gr_sto

f_gr_sto = build_gr_inverse_map()

# Preview reconstructed full-cell voltage for all eSOH scenarios.
plt.figure()
plt.title("Shifted Vfit for Optimized Parameters")
plt.xlabel("Shifted Q (Q@4.2V = 0, Q@2.9V = 2)")
plt.ylabel("Vfit")
plt.grid(True)

all_exports = {}

for res, label in zip(optimized_res_list, labels):
    Cn_Si, Cn_Gr, x100, Cp, y100, si_scale_a, si_shift_b = map(float, res)

    # Evaluate full-cell voltage and electrode potentials on the common Q grid.
    Vfit_exp = np.array(
        [ocp_3(res, Q, use_reconstructed_si_ocp=(SI_RECONSTRUCTION_MODE == "reconstructed"))[0]
         for Q in Qdata_expand],
        dtype=float,
    )
    Vfit_Anode_exp = np.array(
        [ocp_3(res, Q, use_reconstructed_si_ocp=(SI_RECONSTRUCTION_MODE == "reconstructed"))[1]
         for Q in Qdata_expand],
        dtype=float,
    )
    Vfit_Cathode_exp = np.array(
        [ocp_3(res, Q, use_reconstructed_si_ocp=(SI_RECONSTRUCTION_MODE == "reconstructed"))[2]
         for Q in Qdata_expand],
        dtype=float,
    )

    # Mask electrode potentials outside their valid OCP windows.
    cathode_valid = np.isfinite(Vfit_Cathode_exp) & (Vfit_Cathode_exp >= CATHODE_P_MIN) & (Vfit_Cathode_exp <= CATHODE_P_MAX)
    anode_valid   = np.isfinite(Vfit_Anode_exp)   & (Vfit_Anode_exp   >= ANODE_P_MIN)   & (Vfit_Anode_exp   <= ANODE_P_MAX)

    # The full-cell trace is valid only where both electrode potentials are valid.
    full_valid = cathode_valid & anode_valid & np.isfinite(Vfit_exp)

    Vfit_Cathode_exp = apply_nan_mask(Vfit_Cathode_exp, cathode_valid)
    Vfit_Anode_exp   = apply_nan_mask(Vfit_Anode_exp, anode_valid)
    Vfit_exp         = apply_nan_mask(Vfit_exp, full_valid)

    # Smooth V(Q) and compute dV/dQ on each valid trace.
    q_full, _, dVdQ_fit_full_native = smooth_then_grad(
        Qdata_expand, Vfit_exp, window_length=win, polyorder=poly
    )
    q_an, V_fit_Anode_exp_smooth, dVdQ_fit_Anode_native = smooth_then_grad(
        Qdata_expand, Vfit_Anode_exp, window_length=win, polyorder=poly
    )
    q_ca, _, dVdQ_fit_Cathode_native = smooth_then_grad(
        Qdata_expand, Vfit_Cathode_exp, window_length=win, polyorder=poly
    )

    # Align derivatives back to the common Q grid.
    dVdQ_fit_exp = np.interp(Qdata_expand, q_full, dVdQ_fit_full_native, left=np.nan, right=np.nan)
    V_fit_Anode_exp = np.interp(Qdata_expand, q_an, V_fit_Anode_exp_smooth, left=np.nan, right=np.nan)
    dVdQ_fit_Anode_exp = np.interp(Qdata_expand, q_an, dVdQ_fit_Anode_native, left=np.nan, right=np.nan)
    dVdQ_fit_Cathode_exp = np.interp(Qdata_expand, q_ca, dVdQ_fit_Cathode_native, left=np.nan, right=np.nan)

    # Preserve the physical validity masks after interpolation.
    dVdQ_fit_exp = apply_nan_mask(dVdQ_fit_exp, full_valid)
    V_fit_Anode_exp = apply_nan_mask(V_fit_Anode_exp, anode_valid)
    dVdQ_fit_Anode_exp = apply_nan_mask(dVdQ_fit_Anode_exp, anode_valid)
    dVdQ_fit_Cathode_exp = apply_nan_mask(dVdQ_fit_Cathode_exp, cathode_valid)

    # Map common anode potential to silicon and graphite stoichiometry.
    si_ocp_eff, f_si_sto = build_si_inverse_map(
        si_scale_a=si_scale_a,
        si_shift_b=si_shift_b,
        reconstruction_mode=SI_RECONSTRUCTION_MODE,
    )

    # OCP supports used for diagnostics and inverse-map checks.
    x_si_ref = np.asarray(si_ocp_eff["sto"], float)
    p_si = np.asarray(si_ocp_eff["p"], float)

    p_gr = np.asarray(Gr_OCP_smooth["p_smooth"], float)
    x_gr_ref = np.asarray(Gr_OCP_smooth["sto"], float)

    x_si_on_common = np.asarray(f_si_sto(V_fit_Anode_exp), float)
    x_gr_on_common = np.asarray(f_gr_sto(V_fit_Anode_exp), float)

    # Apply the same anode validity mask to material stoichiometries.
    x_si_on_common = apply_nan_mask(x_si_on_common, anode_valid)
    x_gr_on_common = apply_nan_mask(x_gr_on_common, anode_valid)

    Q_si = x_si_on_common * Cn_Si
    Q_gr = x_gr_on_common * Cn_Gr

    # Compute material-resolved dV/dQ on native Si and Gr capacity axes.
    q_si_native, _, dVdQ_fit_si_native = smooth_then_grad(
        Q_si, Vfit_exp, window_length=win, polyorder=poly
    )
    q_gr_native, _, dVdQ_fit_gr_native = smooth_then_grad(
        Q_gr, Vfit_exp, window_length=win, polyorder=poly
    )

    # Project material-resolved derivatives back onto the common full-cell Q grid.
    dVdQ_fit_si_exp = np.full_like(Qdata_expand, np.nan, dtype=float)
    dVdQ_fit_gr_exp = np.full_like(Qdata_expand, np.nan, dtype=float)

    mask_si = np.isfinite(Q_si) & full_valid
    if len(q_si_native) >= 5 and np.isfinite(dVdQ_fit_si_native).any():
        qsi_u, dsi_u = _unique_sorted_xy(q_si_native, dVdQ_fit_si_native)
        if len(qsi_u) >= 5:
            dVdQ_fit_si_exp[mask_si] = np.interp(
                Q_si[mask_si],
                qsi_u,
                dsi_u,
                left=np.nan,
                right=np.nan,
            )

    mask_gr = np.isfinite(Q_gr) & full_valid
    if len(q_gr_native) >= 5 and np.isfinite(dVdQ_fit_gr_native).any():
        qgr_u, dgr_u = _unique_sorted_xy(q_gr_native, dVdQ_fit_gr_native)
        if len(qgr_u) >= 5:
            dVdQ_fit_gr_exp[mask_gr] = np.interp(
                Q_gr[mask_gr],
                qgr_u,
                dgr_u,
                left=np.nan,
                right=np.nan,
            )

    dVdQ_fit_si_exp = apply_nan_mask(dVdQ_fit_si_exp, full_valid)
    dVdQ_fit_gr_exp = apply_nan_mask(dVdQ_fit_gr_exp, full_valid)

    # Convert dV/dQ to incremental-capacity traces dQ/dV.
    dqdv_fit = inv_or_nan(dVdQ_fit_exp)
    dqdv_fit_anode = inv_or_nan(dVdQ_fit_Anode_exp)
    dqdv_fit_cathode = inv_or_nan(dVdQ_fit_Cathode_exp)
    dqdv_fit_si = inv_or_nan(dVdQ_fit_si_exp)
    dqdv_fit_gr = inv_or_nan(dVdQ_fit_gr_exp)

    dqdv_fit = apply_nan_mask(dqdv_fit, full_valid)
    dqdv_fit_anode = apply_nan_mask(dqdv_fit_anode, anode_valid)
    dqdv_fit_cathode = apply_nan_mask(dqdv_fit_cathode, cathode_valid)
    dqdv_fit_si = apply_nan_mask(dqdv_fit_si, full_valid)
    dqdv_fit_gr = apply_nan_mask(dqdv_fit_gr, full_valid)

    # Print inverse-map coverage diagnostics for reproducibility.
    print(f"--- {label} ---")
    print("anode potential range:", np.nanmin(V_fit_Anode_exp), np.nanmax(V_fit_Anode_exp))
    print("Si OCP p-range:", np.nanmin(p_si), np.nanmax(p_si))
    print("Gr OCP p-range:", np.nanmin(p_gr), np.nanmax(p_gr))
    print("finite x_si / x_gr:", np.sum(np.isfinite(x_si_on_common)), np.sum(np.isfinite(x_gr_on_common)))
    print("finite Q_si / Q_gr:", np.sum(np.isfinite(Q_si)), np.sum(np.isfinite(Q_gr)))
    print("finite dqdv_fit_si / dqdv_fit_gr:", np.sum(np.isfinite(dqdv_fit_si)), np.sum(np.isfinite(dqdv_fit_gr)))

    # Define the full-cell SoC axis using the 2.9 V and 4.2 V references.
    mask = np.isfinite(Vfit_exp)
    Vfit_clean = Vfit_exp[mask]
    Qin_clean = Qdata_expand[mask]

    idx_4_2 = np.argmin(np.abs(Vfit_clean - 4.2))
    idx_2_9 = np.argmin(np.abs(Vfit_clean - 2.9))
    Q_ref_4_2 = float(Qin_clean[idx_4_2])
    Q_ref_2_9 = float(Qin_clean[idx_2_9])

    Q_shifted = -(Qdata_expand - Q_ref_2_9)
    SoCs = (Qdata_expand - Q_ref_2_9) / (-Q_ref_2_9 + Q_ref_4_2)

    # Add this scenario to the preview voltage plot.
    plt.plot(SoCs, Vfit_exp, label=label)

    # Export full-cell, electrode, and material-resolved traces.
    
    eSoH_details_df = pd.DataFrame({
        'Q_exp': Q_shifted,
        'V_fit': Vfit_exp,
        'dqdv_fit': dqdv_fit,
        'dvdq_fit': dVdQ_fit_exp,

        'V_fit_anode': V_fit_Anode_exp,
        'dqdv_fit_anode': dqdv_fit_anode,
        'dvdq_fit_anode': dVdQ_fit_Anode_exp,

        'V_fit_cathode': Vfit_Cathode_exp,
        'dqdv_fit_cathode': dqdv_fit_cathode,
        'dvdq_fit_cathode': dVdQ_fit_Cathode_exp,

        'dqdv_fit_si': dqdv_fit_si,
        'dqdv_fit_gr': dqdv_fit_gr,

        'SoCs': SoCs,
        'Q_si': Q_si,
        'Q_gr': Q_gr,

        # Material stoichiometries projected onto the full-cell SoC axis.
        'x_si': x_si_on_common,
        'x_gr': x_gr_on_common,
        
    })


    file_path = eSOH_export_DIR / f"{label}_eSOH_details_70EOL.csv"
    eSoH_details_df.to_csv(file_path, index=False)

    # Export the BOL and effective aged silicon OCP curves for comparison.
    # Baseline silicon OCP.
    x_si_orig = np.asarray(Si_OCP["sto"], float)
    p_si_orig = np.asarray(Si_OCP["p"], float)

    # Effective silicon OCP after the fitted scale and shift deformation.
    x_si_recon = np.asarray(si_ocp_eff["sto"], float)
    p_si_recon = np.asarray(si_ocp_eff["p"], float)

    # Interpolate both silicon OCP curves onto a common stoichiometry grid.
    sto_common = np.linspace(0.0, 1.0, 1000)

    # Baseline silicon OCP on common x grid.
    x1, y1 = _unique_sorted_xy(x_si_orig, p_si_orig)
    p_orig_on_common = np.interp(sto_common, x1, y1, left=np.nan, right=np.nan)

    # Effective silicon OCP on common x grid.
    x2, y2 = _unique_sorted_xy(x_si_recon, p_si_recon)
    p_recon_on_common = np.interp(sto_common, x2, y2, left=np.nan, right=np.nan)

    # Differential silicon OCP on the common x grid.
    dUdx_si_original = np.gradient(p_orig_on_common, sto_common)
    dUdx_si_reconstructed_deformed = np.gradient(p_recon_on_common, sto_common)

    # Reconstruct the composite anode OCP implied by the deformed Si curve.
    p_common = np.asarray(Gr_OCP_smooth["p_smooth"], float)
    x_gr = np.asarray(Gr_OCP_smooth["sto"], float)

    # Map silicon stoichiometry to the graphite potential axis before mixing.
    order_si = np.argsort(p_si_recon)
    p_si_sorted = p_si_recon[order_si]
    x_si_sorted = x_si_recon[order_si]
    p_si_sorted, idx_si = np.unique(p_si_sorted, return_index=True)
    x_si_sorted = x_si_sorted[idx_si]

    x_si_on_p_common = np.interp(
        p_common,
        p_si_sorted,
        x_si_sorted,
        left=float(x_si_sorted[0]),
        right=float(x_si_sorted[-1]),
    )

    alpha = Cn_Si / (Cn_Si + Cn_Gr)
    x_anode = alpha * x_si_on_p_common + (1.0 - alpha) * x_gr

    # Invert composite x(U) to obtain U_anode(x).
    x_anode_u, p_anode_u = _unique_sorted_xy(x_anode, p_common)

    x_anode_common = np.linspace(
        float(np.nanmin(x_anode_u)),
        float(np.nanmax(x_anode_u)),
        1000
    )
    p_anode_reconstructed = np.interp(
        x_anode_common,
        x_anode_u,
        p_anode_u,
        left=np.nan,
        right=np.nan
    )
    dUdx_anode_reconstructed = np.gradient(p_anode_reconstructed, x_anode_common)

    si_ocp_compare_df = pd.DataFrame({
        # Silicon OCP comparison on a common stoichiometry axis.
        "sto_common": sto_common,
        "p_si_original": p_orig_on_common,
        "p_si_reconstructed_deformed": p_recon_on_common,
        "si_scale_a": np.full_like(sto_common, si_scale_a, dtype=float),
        "si_shift_b": np.full_like(sto_common, si_shift_b, dtype=float),

        # Differential OCP and reconstructed composite-anode OCP.
        "dUdx_si_original": dUdx_si_original,
        "dUdx_si_reconstructed_deformed": dUdx_si_reconstructed_deformed,
        "x_anode_common": x_anode_common,
        "p_anode_reconstructed": p_anode_reconstructed,
        "dUdx_anode_reconstructed": dUdx_anode_reconstructed,
    })

    si_ocp_compare_path = eSOH_export_DIR / f"{label}_Si_OCP_original_vs_reconstructed_deformed.csv"
    si_ocp_compare_df.to_csv(si_ocp_compare_path, index=False)
    
    # Also save the raw-axis silicon curves without interpolation.
    n_raw = max(len(x_si_orig), len(x_si_recon))

    def _pad(arr, n):
        arr = np.asarray(arr, float)
        if len(arr) >= n:
            return arr[:n]
        out = np.full(n, np.nan, dtype=float)
        out[:len(arr)] = arr
        return out

    si_ocp_compare_raw_df = pd.DataFrame({
        "sto_original_raw": _pad(x_si_orig, n_raw),
        "p_original_raw": _pad(p_si_orig, n_raw),
        "sto_reconstructed_raw": _pad(x_si_recon, n_raw),
        "p_reconstructed_raw": _pad(p_si_recon, n_raw),
    })

    si_ocp_compare_raw_path = eSOH_export_DIR / f"{label}_Si_OCP_original_vs_reconstructed_deformed_rawaxis.csv"
    si_ocp_compare_raw_df.to_csv(si_ocp_compare_raw_path, index=False)
    
    all_exports[label] = eSoH_details_df

    print(f"--- {label} ---")
    print("all lengths:", len(Q_shifted), len(Vfit_exp), len(dVdQ_fit_exp), len(dVdQ_fit_si_exp), len(dVdQ_fit_gr_exp))
    print("finite full/anode/cathode:", np.sum(np.isfinite(Vfit_exp)), np.sum(np.isfinite(V_fit_Anode_exp)), np.sum(np.isfinite(Vfit_Cathode_exp)))
    print("finite Q_si:", np.sum(np.isfinite(Q_si)))
    print("Q refs:", [Q_ref_2_9, Q_ref_4_2])
    
    print("anode potential range:", np.nanmin(V_fit_Anode_exp), np.nanmax(V_fit_Anode_exp))
    print("Si OCP p-range:", np.nanmin(p_si), np.nanmax(p_si))
    print("Gr OCP p-range:", np.nanmin(p_gr), np.nanmax(p_gr))

plt.legend()
plt.tight_layout()
plt.show()

# Export reference OCP derivative data

This final export saves graphite and NMC reference OCP curves and their derivatives. These tables support the OCP and differential-OCP panels used to interpret why silicon dominates current at low SoC.

In [ ]:
# Export graphite and NMC OCP reference curves and derivatives.
x_gr_ref = np.asarray(Gr_OCP_smooth["sto"], float)
p_gr_ref = np.asarray(Gr_OCP_smooth["p_smooth"], float)

x_nmc_ref = np.asarray(NMC_OCP["sto"], float)
p_nmc_ref = np.asarray(NMC_OCP["p"], float)

x_gr_u, p_gr_u = _unique_sorted_xy(x_gr_ref, p_gr_ref)
x_nmc_u, p_nmc_u = _unique_sorted_xy(x_nmc_ref, p_nmc_ref)

# dU/dx on each material stoichiometry grid.
dUdx_gr = np.gradient(p_gr_u, x_gr_u)
dUdx_nmc = np.gradient(p_nmc_u, x_nmc_u)

# Graphite dx/dU on a uniform 1 mV potential grid.
p_gr_sort_idx = np.argsort(p_gr_u)
p_gr_for_inv = p_gr_u[p_gr_sort_idx]
x_gr_for_inv = x_gr_u[p_gr_sort_idx]

# Remove duplicate potential values before inverse mapping.
p_gr_for_inv, idx_gr_p = np.unique(p_gr_for_inv, return_index=True)
x_gr_for_inv = x_gr_for_inv[idx_gr_p]

# Uniform 1 mV graphite potential grid.
p_gr_1mV = np.arange(
    float(np.min(p_gr_for_inv)),
    float(np.max(p_gr_for_inv)) + 0.001,
    0.001
)

# Graphite x(U) on the 1 mV grid.
x_gr_on_1mV = np.interp(p_gr_1mV, p_gr_for_inv, x_gr_for_inv, left=np.nan, right=np.nan)

# Graphite dx/dU on the 1 mV grid.
dxdU_gr_1mV = np.gradient(x_gr_on_1mV, p_gr_1mV)

# NMC dx/dU on the same style of 1 mV potential grid.
p_nmc_sort_idx = np.argsort(p_nmc_u)
p_nmc_for_inv = p_nmc_u[p_nmc_sort_idx]
x_nmc_for_inv = x_nmc_u[p_nmc_sort_idx]

p_nmc_for_inv, idx_nmc_p = np.unique(p_nmc_for_inv, return_index=True)
x_nmc_for_inv = x_nmc_for_inv[idx_nmc_p]

p_nmc_1mV = np.arange(
    float(np.min(p_nmc_for_inv)),
    float(np.max(p_nmc_for_inv)) + 0.001,
    0.001
)

x_nmc_on_1mV = np.interp(p_nmc_1mV, p_nmc_for_inv, x_nmc_for_inv, left=np.nan, right=np.nan)
dxdU_nmc_1mV = np.gradient(x_nmc_on_1mV, p_nmc_1mV)

# Pack variable-length reference arrays into one table.
n_ref = max(len(x_gr_u), len(x_nmc_u), len(p_gr_1mV), len(p_nmc_1mV))

def _pad(arr, n):
    arr = np.asarray(arr, float)
    out = np.full(n, np.nan, dtype=float)
    out[:min(len(arr), n)] = arr[:min(len(arr), n)]
    return out

ref_ocp_df = pd.DataFrame({
    "x_gr": _pad(x_gr_u, n_ref),
    "p_gr": _pad(p_gr_u, n_ref),
    "dUdx_gr": _pad(dUdx_gr, n_ref),

    "x_nmc": _pad(x_nmc_u, n_ref),
    "p_nmc": _pad(p_nmc_u, n_ref),
    "dUdx_nmc": _pad(dUdx_nmc, n_ref),

    # Graphite inverse-grid export.
    "p_gr_1mV": _pad(p_gr_1mV, n_ref),
    "x_gr_1mV": _pad(x_gr_on_1mV, n_ref),
    "dxdU_gr_1mV": _pad(dxdU_gr_1mV, n_ref),

    # NMC inverse-grid export.
    "p_nmc_1mV": _pad(p_nmc_1mV, n_ref),
    "x_nmc_1mV": _pad(x_nmc_on_1mV, n_ref),
    "dxdU_nmc_1mV": _pad(dxdU_nmc_1mV, n_ref),
})

ref_ocp_path = eSOH_export_DIR / "Graphite_NMC_OCP_reference.csv"
ref_ocp_df.to_csv(ref_ocp_path, index=False)
print(f"Saved: {ref_ocp_path}")